In [ ]:
# =========================================================
# TASK 5 - AUTO TAGGING SUPPORT TICKETS USING LLM
# =========================================================

# =========================================================
# STEP 1 - IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# HuggingFace Pipeline
from transformers import pipeline

c:\Users\Muhammad Ahsaan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# =========================================================
# STEP 2 - LOAD DATASET
# =========================================================

# Load support ticket dataset
df = pd.read_csv("support_tickets.csv")

# Show first rows
print(df.head())

# =========================================================
# STEP 3 - DATA EXPLORATION
# =========================================================

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# =========================================================
# STEP 4 - DEFINE POSSIBLE TAGS
# =========================================================

# Categories for ticket tagging

candidate_labels = [
    "Billing Issue",
    "Technical Support",
    "Account Access",
    "Refund Request",
    "Password Reset",
    "Delivery Problem",
    "Subscription Issue",
    "General Inquiry"
]

# =========================================================
# STEP 5 - LOAD ZERO-SHOT CLASSIFICATION MODEL
# =========================================================

# Load transformer pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("\nLLM Model Loaded Successfully!")

In [ ]:
# =========================================================
# STEP 6 - TEST SINGLE TICKET
# =========================================================

sample_ticket = """
I am unable to login into my account
even after resetting the password.
"""

result = classifier(
    sample_ticket,
    candidate_labels
)

print("\nSingle Ticket Prediction:")
print(result)

# =========================================================
# STEP 7 - FUNCTION FOR TOP 3 TAGS
# =========================================================

def get_top_3_tags(ticket_text):

    # Run classification
    result = classifier(
        ticket_text,
        candidate_labels
    )

    # Extract top 3 labels
    top_3_labels = result['labels'][:3]

    # Extract scores
    top_3_scores = result['scores'][:3]

    return list(zip(top_3_labels, top_3_scores))

# =========================================================
# STEP 8 - APPLY MODEL TO DATASET
# =========================================================

# Store predictions
top_predictions = []

# Loop through support tickets
for ticket in df['ticket_text']:

    predictions = get_top_3_tags(ticket)

    top_predictions.append(predictions)

In [ ]:
# =========================================================
# STEP 9 - STORE RESULTS
# =========================================================

# Top 1 tag
df['Top_Tag'] = [
    preds[0][0]
    for preds in top_predictions
]

# Top 1 confidence
df['Top_Score'] = [
    preds[0][1]
    for preds in top_predictions
]

# Top 3 tags
df['Top_3_Tags'] = top_predictions

# =========================================================
# STEP 10 - SHOW RESULTS
# =========================================================

print("\nPrediction Results:")

print(
    df[
        [
            'ticket_text',
            'Top_Tag',
            'Top_Score'
        ]
    ].head()
)

In [ ]:
# =========================================================
# STEP 11 - VISUALIZATION
# =========================================================

# Count predicted tags

plt.figure(figsize=(10,6))

sns.countplot(
    y=df['Top_Tag'],
    order=df['Top_Tag'].value_counts().index
)

plt.title("Predicted Support Ticket Categories")

plt.xlabel("Count")

plt.ylabel("Category")

plt.show()

In [ ]:
# =========================================================
# STEP 12 - ZERO-SHOT vs FEW-SHOT EXPLANATION
# =========================================================

print("\nZERO-SHOT LEARNING")
print("--------------------------------")
print("The model predicts categories without task-specific training.")

print("\nFEW-SHOT LEARNING")
print("--------------------------------")
print("Few-shot learning improves predictions by providing examples in prompts.")

# =========================================================
# STEP 13 - FEW-SHOT PROMPT EXAMPLES
# =========================================================

few_shot_examples = """

Example 1:
Ticket: My payment failed twice.
Tag: Billing Issue

Example 2:
Ticket: I forgot my account password.
Tag: Password Reset

Example 3:
Ticket: My package has not arrived yet.
Tag: Delivery Problem

"""

print("\nFew-Shot Prompt Examples:")
print(few_shot_examples)


In [ ]:

# =========================================================
# STEP 14 - SAVE RESULTS
# =========================================================

# Save tagged tickets
df.to_csv(
    "tagged_support_tickets.csv",
    index=False
)

print("\nTagged tickets saved successfully!")

# =========================================================
# STEP 15 - FINAL INSIGHTS
# =========================================================

print("\nFINAL INSIGHTS")
print("------------------------------------------------")

print("- LLM successfully categorized support tickets.")
print("- Zero-shot learning allowed predictions without custom training.")
print("- Few-shot examples improved understanding of ticket categories.")
print("- Top 3 probable tags were generated for each ticket.")
print("- This project demonstrates practical LLM-based NLP classification.")